In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as stats
from pathlib import Path
import missingno as msno

In [2]:
TITANIC_DATA_PATH   = Path('../practice/datasets/kaggle/titanic/train.csv')
TITANIC_MODEL_DIR   = Path('../practice/models/titanic'); TITANIC_MODEL_DIR.mkdir(exist_ok=True)
TITANIC_OUTPUT_DIR  = Path('../practice/outputs/titanic'); TITANIC_OUTPUT_DIR.mkdir(exist_ok=True)

### how coders or engineers deal with missing values rather than learners in real data in companies at production level


| Layer | Details |
| :--- | :--- |
| **Layer 1 — Data contract & schema validation (at ingestion)** | **Stop bad data before it enters the system**<br>Great Expectations, Pandera, Pydantic, dbt tests, Cerberus<br><br>1. Define expected schema: which columns are nullable, which are required<br>2. Assert max allowed missing % per column at pipeline entry<br>3. Fail loudly or quarantine rows that breach the contract<br>4. Log violation events to an observability platform |
| **Layer 2 — Imputation as a versioned artifact (not ad-hoc code)** | **Imputer is trained, serialized, versioned, and deployed like a model**<br>MLflow, joblib, sklearn, Pipeline, DVC, Feast feature store<br><br>1. Imputer fit on training data, saved as `imputer_v`<br>2. `.pkl` alongside model; versioned in artifact registry — every model version has a paired imputer<br>3. Loaded at inference — never refit on live data<br>4. Rollback possible — revert imputer when model is rolled back |
| **Layer 3 — Runtime drift & missing-rate monitoring** | **Detect when production data starts differing from training distribution**<br>Evidently AI, WhyLogs, Prometheus, Grafana, Datadog<br><br>1. Track missing rate per column on every inference batch<br>2. Alert when live missing rate deviates >10% from training baseline<br>3. Track imputation coverage — how often each strategy fires<br>4. Dashboard: missing rate trend over time per feature |
| **Layer 4 — Incident response & fallback strategies** | **What happens when a critical feature goes entirely missing in production**<br>Circuit breaker, Feature fallback, Model degradation mode, PagerDuty alert<br><br>1. Define critical vs non-critical features — missing critical = halt or fallback model<br>2. Maintain a simpler fallback model trained without the critical feature<br>3. Auto-switch to fallback when critical feature missing rate > threshold<br>4. Page on-call engineer; log incident for post-mortem |


## Layer 1 — Data contracts at ingestion

Engineers don't hope data arrives clean. They enforce a contract at the pipeline boundary and reject or quarantine violations before they corrupt downstream systems.

In [3]:
! pip install pandera

  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandera]


In [5]:
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check
import pandas as pd

# Define the contract once — lives in version control
schema = DataFrameSchema({
    "user_id":      Column(int,   nullable=False),
    "age":          Column(float, nullable=True,
                           checks=Check.in_range(0, 120)),
    "income":       Column(float, nullable=True),
    "credit_score": Column(float, nullable=True,
                           checks=Check.in_range(300, 850)),
    "city":         Column(str,   nullable=True),
    "product_id":   Column(int,   nullable=False),
},
checks=Check(lambda df: df.isna().mean().le(0.30).all(),
             error="A column exceeded 30% missing threshold")
)

def ingest_batch(raw_df: pd.DataFrame, batch_id: str) -> pd.DataFrame:
    try:
        validated = schema.validate(raw_df, lazy=True)
        return validated
    except pa.errors.SchemaErrors as exc:
        failure_cases = exc.failure_cases
        failure_cases.to_parquet(f"quarantine/{batch_id}.parquet")
        print(f"[ALERT] Schema violation in batch {batch_id}:\n{failure_cases}")
        valid_rows = raw_df.drop(
            failure_cases["index"].dropna().astype(int)
        )
        return valid_rows

## Layer 2 — Imputer as a versioned artifact
The imputer is not a script. It is an artifact — trained once, versioned, logged, and loaded at inference exactly like a model.

In [6]:
import mlflow
import mlflow.sklearn
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

# ── Training job (runs once per model version) ─────────────────────
with mlflow.start_run(run_name="churn_model_v3") as run:
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   GradientBoostingClassifier(n_estimators=300)),
    ])

    pipeline.fit(X_train, y_train)
    val_auc = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

    # Log the whole pipeline — imputer state is inside it
    mlflow.sklearn.log_model(pipeline, artifact_path="pipeline")
    mlflow.log_metric("val_auc", val_auc)

    # Also log imputer stats separately for inspection
    imputer = pipeline.named_steps["imputer"]
    mlflow.log_dict(
        {col: val for col, val in
         zip(X_train.columns, imputer.statistics_)},
        "imputer_statistics.json"
    )

    run_id = run.info.run_id
    print(f"Logged run: {run_id}")

2026/05/10 19:29:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/10 19:29:01 INFO mlflow.store.db.utils: Updating database tables


NameError: name 'X_train' is not defined

In [7]:
# ── Inference service (FastAPI endpoint) ───────────────────────────
import mlflow.pyfunc
from fastapi import FastAPI
import pandas as pd

app = FastAPI()

# Load once at startup — never refit
MODEL_URI = "models:/churn_model/Production"
pipeline  = mlflow.sklearn.load_model(MODEL_URI)

@app.post("/predict")
def predict(features: dict):
    df = pd.DataFrame([features])

    # Pipeline handles imputation, scaling, prediction in one call
    # If a feature is missing in the request JSON → NaN → imputed
    proba = pipeline.predict_proba(df)[0, 1]
    return {"churn_probability": round(float(proba), 4)}

MlflowException: Registered Model with name=churn_model not found

## Layer 3 — Missing-rate monitoring in production
This is the part most tutorials skip entirely. Engineers track missing rates continuously, not just once at training time.


In [8]:
import numpy as np
import pandas as pd
from datetime import datetime
from prometheus_client import Gauge, Counter, start_http_server

# Prometheus metrics — scraped by Grafana
missing_rate_gauge = Gauge(
    "feature_missing_rate",
    "Fraction of nulls in the last inference batch",
    ["feature_name"]
)
imputation_fired_counter = Counter(
    "imputation_fired_total",
    "Number of times imputation was applied",
    ["feature_name"]
)
drift_alert_counter = Counter(
    "missing_rate_drift_alerts_total",
    "Number of drift alerts triggered",
    ["feature_name"]
)

class ProductionMissingnessMonitor:
    def __init__(self,
                 train_df: pd.DataFrame,
                 alert_threshold: float = 0.10,
                 critical_features: list[str] = None):
        # Baseline computed from training data — never updated
        self.baseline     = train_df.isna().mean().to_dict()
        self.threshold    = alert_threshold
        self.critical     = set(critical_features or [])
        self.history: list[dict] = []

    def check_batch(self, batch_df: pd.DataFrame) -> dict:
        results = {"timestamp": datetime.utcnow().isoformat(),
                   "n_rows": len(batch_df),
                   "alerts": [], "critical_alerts": []}

        for col in batch_df.columns:
            live_rate  = float(batch_df[col].isna().mean())
            train_rate = self.baseline.get(col, 0.0)
            drift      = abs(live_rate - train_rate)

            # Push to Prometheus
            missing_rate_gauge.labels(feature_name=col).set(live_rate)
            if live_rate > 0:
                imputation_fired_counter.labels(feature_name=col).inc(
                    int(batch_df[col].isna().sum()))

            if drift > self.threshold:
                alert = {
                    "feature":    col,
                    "train_rate": round(train_rate, 4),
                    "live_rate":  round(live_rate, 4),
                    "drift":      round(drift, 4),
                    "severity":   "critical" if col in self.critical else "warning"
                }
                results["alerts"].append(alert)
                drift_alert_counter.labels(feature_name=col).inc()

                if col in self.critical:
                    results["critical_alerts"].append(alert)

        self.history.append(results)
        return results

    def should_halt(self, check_result: dict) -> bool:
        """Return True if any critical feature has >80% missing."""
        for alert in check_result["critical_alerts"]:
            if alert["live_rate"] > 0.80:
                return True
        return False

# ── Usage in inference loop ────────────────────────────────────────
monitor = ProductionMissingnessMonitor(
    train_df=X_train,
    alert_threshold=0.10,
    critical_features=["credit_score", "account_age_days"]
)

def run_inference_batch(batch_df: pd.DataFrame):
    check = monitor.check_batch(batch_df)

    if monitor.should_halt(check):
        # Switch to fallback model — see Layer 4
        return fallback_model.predict(batch_df)

    if check["alerts"]:
        notify_slack(
            channel="#ml-alerts",
            message=format_alert(check["alerts"])
        )

    return pipeline.predict_proba(batch_df)

NameError: name 'X_train' is not defined

## Layer 4 — Fallback model and circuit breaker
When a critical feature goes missing entirely (upstream pipeline breaks, API goes down), the production model degrades gracefully instead of crashing or returning garbage.

In [9]:
import joblib
from enum import Enum

class ModelMode(Enum):
    FULL    = "full"        # all features available
    REDUCED = "reduced"     # critical features missing, use fallback
    HALTED  = "halted"      # too degraded to serve — return default

class ResilientPredictor:
    """
    Wraps full model + fallback model.
    Automatically selects which to use based on feature availability.
    """
    def __init__(self,
                 full_pipeline,
                 reduced_pipeline,      # trained WITHOUT critical features
                 critical_features: list[str],
                 missing_halt_threshold: float = 0.95):
        self.full       = full_pipeline
        self.reduced    = reduced_pipeline
        self.critical   = critical_features
        self.halt_thr   = missing_halt_threshold
        self.mode       = ModelMode.FULL

    def _assess_mode(self, df: pd.DataFrame) -> ModelMode:
        for col in self.critical:
            if col not in df.columns:
                return ModelMode.REDUCED
            missing_rate = df[col].isna().mean()
            if missing_rate > self.halt_thr:
                return ModelMode.HALTED
            if missing_rate > 0.5:
                return ModelMode.REDUCED
        return ModelMode.FULL

    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:
        mode = self._assess_mode(df)
        self.mode = mode  # expose for monitoring

        if mode == ModelMode.FULL:
            return self.full.predict_proba(df)

        elif mode == ModelMode.REDUCED:
            # Drop critical features — fallback was trained without them
            safe_cols = [c for c in df.columns if c not in self.critical]
            return self.reduced.predict_proba(df[safe_cols])

        else:  # HALTED
            # Return population base rate — least harmful default
            n = len(df)
            base_rate = 0.12   # e.g. known 12% churn rate
            return np.full((n, 2), [1 - base_rate, base_rate])

# ── Training both models upfront ───────────────────────────────────
# Full model
full_pipe = Pipeline([...]).fit(X_train, y_train)

# Reduced model — trained without critical features
critical = ["credit_score", "account_age_days"]
X_reduced = X_train.drop(columns=critical)
reduced_pipe = Pipeline([...]).fit(X_reduced, y_train)

# Package together
predictor = ResilientPredictor(full_pipe, reduced_pipe, critical)
joblib.dump(predictor, "resilient_predictor_v3.pkl")

NameError: name 'X_train' is not defined

### What engineers do differently — the mindset gap


| Learner mindset | Engineer mindset |
| :--- | :--- |
| Impute once in a notebook | Impute inside a versioned Pipeline artifact |
| Fix NaNs before training | Define nullable columns in a schema contract |
| Notice missing data when model breaks | Monitor missing rate continuously per feature |
| Re-run notebook to fix | Circuit breaker switches to fallback automatically |
| `df.fillna(df.mean())` | `imputer.fit(X_train); imputer.transform(X_live)` |
| Doesn't track what was imputed | Prometheus metrics on imputation rate per feature |
| One model | Full model + reduced fallback model |
| git commit "fixed missing values" | Imputer versioned alongside model in MLflow registry |
